# 06 — Final report

The final, self-contained synthesis of the QL2 Sixt Canada rate-feed analysis
(data-plan Section 7). It runs the validated pricing pipeline **exactly once**
and presents the readiness evidence, matched-location pricing, price changes,
visible assortment and monitoring of that one run, followed by the
assumptions, exclusions, limitations, open questions, data requests and
conclusions.

It is a thin presentation layer over `ql2_sixt_canada_analysis.final_report`
(`run_final_report` → `run_pricing_pipeline` → the existing `*_from_pipeline`
analyses). It does not depend on notebooks 01 to 05 or on earlier kernel state.
Run it top to bottom after restarting the kernel.

## Purpose and analytical questions

Can QL2 trust this hourly Sixt Canada rate feed enough to support customer
decisions, and which pricing and visible-assortment signals can be extracted
once coverage, timestamps, locations and product identity are controlled?

1. **Coverage and reliability.** Did each scheduled collection return the
   expected locations and products at the expected cadence, with consistent
   identifiers, timestamps, counts and attributes?
2. **Pricing structure.** For the same vehicle and collection event, how do
   listed prices differ between airport and downtown locations by city and
   vehicle class?
3. **Change and anomaly detection.** When prices or visible assortment change,
   are the movements isolated, synchronized and persistent, and do they look
   more like repricing or a collection issue?
4. **Actionability.** Which gaps or anomalies warrant investigation, which
   monitoring rules could detect them, and what more data would a pricing
   decision need?

## Scope and confidentiality

All QL2 data is proprietary. This notebook shows only status values, blocker
categories, approved configuration keys, sanitized aggregate tables, in-memory
figures and generated interpretation text. It never shows offer-level rows,
product identities, individual prices, identifiers, raw location labels,
source file names, paths or the detailed event table. Nothing is written.

Supplier, channel and win/meet/loss questions are out of scope: the files hold
one supplier and one apparent source mode. Clear all outputs before committing
(Edit ▸ Clear Outputs of All Cells).

In [ ]:
from IPython.display import Image, display

from ql2_sixt_canada_analysis import (
    ReportSection,
    data_requests_table,
    final_conclusions,
    final_section_table,
    final_status_lines,
    heatmap_png,
    interpret_section,
    matched_premium_png,
    matched_summary_table,
    monitoring_overview_table,
    open_questions_table,
    run_final_report,
)
from ql2_sixt_canada_analysis.paths import resolve_raw_data_dir

# Raw-data directory: the centralized default unless the documented environment
# override is set. Set RAW_DIR_OVERRIDE only for a deliberate local experiment.
RAW_DIR_OVERRIDE = None

raw_dir = resolve_raw_data_dir(RAW_DIR_OVERRIDE)
print("Raw-data directory selected.")

## Data and pipeline readiness

One call runs the pricing pipeline once and passes that same result to every
analysis. Readiness combines the approved source-stream contract, per-stream
schedule coverage (with the one governed Calgary exclusion), job linkage,
reconciliation, the trusted join, city integrity, temporal and rental-date
rules, the pricing-eligible population, vehicle-attribute stability, the
location authority and the canonical-offer combination. If the evidence is
blocked or not bound to this run, only blocker categories are shown and no
commercial finding is presented as valid. The table has one row per section.

In [ ]:
report = run_final_report(raw_dir)
for line in final_status_lines(report):
    print(line)
display(final_section_table(report))

In [ ]:
print(interpret_section(report, ReportSection.READINESS))

## Matched-location pricing findings

One matched pair is one approved city, one collection job, one rental period,
one exact product, one currency and price basis, with exactly one airport and
one canonical downtown offer (Vancouver Thurlow combined into Vancouver
Downtown). Premium = airport minus downtown; percentages are of the downtown
price and unavailable for a zero downtown price.

In [ ]:
if report.findings_valid(ReportSection.MATCHED_PRICING):
    display(matched_summary_table(report))
    display(Image(data=matched_premium_png(report)))
else:
    print("Skipped: matched-location pricing is blocked.")

In [ ]:
print(interpret_section(report, ReportSection.MATCHED_PRICING))

## Price-change findings

Each offer is compared with the immediately preceding eligible capture of the
same canonical location, one hour earlier. The persistence table judges each
increase or decrease at the next eligible capture; the heatmap shows
increases and decreases separately for every location and hour.

In [ ]:
if report.findings_valid(ReportSection.PRICE_CHANGES):
    display(report.price_changes.tables.persistence_summary)
    display(Image(data=heatmap_png(report.price_changes.analysis)))
else:
    print("Skipped: the price-change presentation is blocked.")

In [ ]:
print(interpret_section(report, ReportSection.PRICE_CHANGES))

## Visible-assortment findings

Visible assortment is the set of products the collection returned for each
canonical location and capture. The location summary gives additions,
removals, retention and Jaccard similarity over valid one-hour intervals;
the second table groups observed drops by scheduled period.

In [ ]:
if report.findings_valid(ReportSection.VISIBLE_ASSORTMENT):
    display(report.assortment.tables.location_summary)
    display(report.assortment.tables.cross_location_drops)
else:
    print("Skipped: the visible-assortment presentation is blocked.")

In [ ]:
print(interpret_section(report, ReportSection.VISIBLE_ASSORTMENT))

## Monitoring and actionability

The eight data-plan Section 6 controls evaluated on the same run, one row per
control: catalog severity, effective severity, calibration status, evaluation
status and typed finding and evidence-gap codes. Contract-based controls can
pass or trigger now; candidate rules have no approved threshold.

In [ ]:
display(monitoring_overview_table(report))

In [ ]:
print(interpret_section(report, ReportSection.MONITORING))

## Assumptions

Approved external decisions (pricing-authority record v8) are kept separate
from these analytical assumptions; the full list, with scope and consequence,
is in `docs/assumptions_exclusions_and_open_questions.md`.

- Listed prices are compared as returned (exact cents, with the currency marker
  and price basis parsed from the price text); they are not total customer cost.
- A product is the approved exact identity (name, type, transmission, seats,
  bags); nothing is matched by similarity.
- Hourly captures of one product are repeated measurements, so statistical
  results are associational.
- Changes are compared only between consecutive eligible captures one hour
  apart, and persistence only at the next eligible capture.
- The material synchronized-movement rule is a descriptive selection rule, not
  an approved threshold.
- Visible assortment is what the collection returned, not supplier
  availability.

## Exclusions

- **Governed:** the one incomplete Calgary parent capture of record v8 stays in
  the raw and audit populations and is excluded from the pricing-eligible
  population; both Calgary stream-periods of that capture are analytically
  null, and no interval spans it.
- **Mechanical:** completely blank rows are removed and counted; partially
  populated rows are kept and evaluated.
- **Invalid or blocked:** unmatched, ambiguous, untrusted, temporally invalid,
  rental-date-ineligible and schedule-ineligible records never enter a
  conclusion that needs trusted pricing evidence.
- **Unassessable:** zero or invalid denominators stay unavailable (never
  infinity or zero); right-censored final-window events are
  confirmation-required and excluded from persistence conclusions.
- **Presentation and privacy:** offer-level rows, identifiers, prices, raw
  labels, file names and paths are never shown; small magnitude groups are
  suppressed.

## Limitations

- The sample spans roughly 90 hours: it is descriptive and cannot establish
  seasonality, long-term behaviour or calibrated thresholds.
- One supplier and one apparent source mode: no supplier, channel or
  win/meet/loss comparison is possible.
- Without collection logs or source snapshots, synchronized movements and
  assortment drops cannot be classified as repricing or as collection defects.
- Matched premiums are associational; no causal claim is made, and capture-time
  and price-percentile views are not presented.
- No production alerting, scheduling, notification, persistence or dashboard
  system exists, and no candidate rule is a production alert.

## Unanswered questions

These questions remain open and are not answered or guessed here. Owners are
named only where a governance record supports them.

In [ ]:
display(open_questions_table())

**Interpretation.** The policy questions block production monitoring, and
the evidence questions block any commercial reading of the change and
assortment candidates. Decisions already resolved in record v8 are not
reopened.

## Requested additional data

Each request names the evidence and the open question it would answer.

In [ ]:
display(data_requests_table())

**Interpretation.** Longer history, the next eligible collections and
collection logs come first: they would confirm or refute the right-censored
observations and separate collection defects from repricing before any
threshold is calibrated.

## Final conclusions

One conclusion per analytical question, generated from the section statuses
above. A blocked section concludes nothing.

In [ ]:
for conclusion in final_conclusions(report):
    print("-", conclusion)

**Interpretation.** Use these conclusions for investigation priorities,
not for pricing decisions: every finding is descriptive for this window and
the pricing-eligible population only.

## Data-plan Section 7 reconciliation

- Remove exploratory clutter: notebooks 01 to 06 were reviewed; stale wording contradicted by record v8 was corrected and each notebook is a thin presentation layer over package APIs.
- Add concise interpretation beneath each result: every result cell is tagged `result` and followed by a cell tagged `interpretation`; this notebook generates its interpretation text from the sanitized reports (`interpret_section`, `final_conclusions`), and blocked sections state what cannot be concluded.
- Document assumptions: summarized above and listed with scope and consequence in `docs/assumptions_exclusions_and_open_questions.md`, separate from the approved decisions of record v8.
- Document exclusions: governed, mechanical, invalid, unassessable and presentation exclusions are summarized above and detailed in the same document.
- Write the data dictionary: `docs/data_dictionary.md` covers every raw field, the derived and presentation fields, grains, keys, relationships and formulas.
- Complete README execution instructions: `README.md` and `notebooks/README.md` give setup, data placement, execution order, validation, output clearing and troubleshooting.
- Record unanswered questions and requested additional data: the two catalogs above, mirrored in the assumptions document.

## Clear all outputs before committing

Use Edit ▸ Clear Outputs of All Cells before committing. `tests/test_notebooks.py`
fails if a committed notebook has outputs or execution counts.

In [ ]:
print("No files written.")